# 08 — Pipelines
**Referencias:** ESL Cap. 7 (data leakage) · Géron Cap. 2

## Por qué Pipelines (Géron Cap. 2)
Sin Pipeline, el preprocesamiento en el train set puede **contaminar** el validation set:
```
# MAL — data leakage:
scaler.fit(X_train); X_train_scaled = scaler.transform(X_train)
# Luego en CV el scaler ya vio todos los folds

# BIEN — Pipeline garantiza fit solo en train:
Pipeline([('sc', StandardScaler()), ('model', LogReg())])
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, OneHotEncoder, PolynomialFeatures
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
import joblib

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# Dataset con mixed types: numéricas + categóricas + missing
n = 2000
df = pd.DataFrame({
    'sessions':        np.random.randint(1, 20, n).astype(float),
    'time_on_site':    np.random.uniform(30, 600, n),
    'pages':           np.random.uniform(1, 8, n),
    'days_since_reg':  np.random.randint(0, 30, n).astype(float),
    'revenue_usd':     np.abs(np.random.normal(50, 40, n)),
    'channel':         np.random.choice(['organic','paid','email','direct'], n),
    'device':          np.random.choice(['mobile','desktop','tablet'], n),
    'plan':            np.random.choice(['free','basic','pro'], n),
})

# Introducir missing values realistas
for col in ['time_on_site','revenue_usd']:
    missing_idx = np.random.choice(n, size=int(n*0.08), replace=False)
    df.loc[missing_idx, col] = np.nan

logit = (-2 + df['sessions'].fillna(0)*0.1 +
         df['time_on_site'].fillna(df['time_on_site'].median())*0.003 +
         df['pages'].fillna(0)*0.15 - df['days_since_reg'].fillna(15)*0.03)
y = (np.random.uniform(0,1,n) < 1/(1+np.exp(-logit))).astype(int)

num_feats = ['sessions','time_on_site','pages','days_since_reg','revenue_usd']
cat_feats  = ['channel','device','plan']
X = df[num_feats + cat_feats]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Missing: {X_train.isnull().sum().sum()} valores')

## 1 — Pipeline básico con ColumnTransformer (Géron Cap. 2)

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_feats),
    ('cat', cat_pipeline, cat_feats),
])

full_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', LogisticRegression(max_iter=1000)),
])

full_pipeline.fit(X_train, y_train)
auc = roc_auc_score(y_test, full_pipeline.predict_proba(X_test)[:,1])
print(f'Pipeline completo — AUC: {auc:.4f}')
print('Pasos del pipeline:', [step[0] for step in full_pipeline.steps])
print('Pasos del preprocessor:', [step[0] for step in preprocessor.transformers])

## 2 — Custom Transformer (Géron Cap. 2)
Para lógica de negocio específica que no está en sklearn.

In [ ]:
class EngagementRatioTransformer(BaseEstimator, TransformerMixin):
    """
    Crea features de ratio basadas en comportamiento del usuario.
    Hereda de BaseEstimator para get/set_params y TransformerMixin para fit_transform.
    """
    def __init__(self, clip_upper=50.0):
        self.clip_upper = clip_upper

    def fit(self, X, y=None):
        # Si hubiera estadísticos a calcular del train set, irían aquí
        return self

    def transform(self, X):
        X = X.copy()
        # pages por sesión
        X['pages_per_session'] = (X['pages'] / (X['sessions'] + 1)).clip(0, self.clip_upper)
        # tiempo por página
        X['time_per_page'] = (X['time_on_site'] / (X['pages'] + 1)).clip(0, 600)
        # revenue por sesión
        X['revenue_per_session'] = (X['revenue_usd'] / (X['sessions'] + 1)).clip(0, 1000)
        return X


class FunctionTransformer(BaseEstimator, TransformerMixin):
    """Envuelve cualquier función Python como paso de Pipeline."""
    def __init__(self, func):
        self.func = func

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return self.func(X)


# Extender num_feats con las features derivadas
extended_num_feats = num_feats + ['pages_per_session', 'time_per_page', 'revenue_per_session']

extended_preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler',  RobustScaler()),
    ]), extended_num_feats),
    ('cat', cat_pipeline, cat_feats),
])

pipeline_with_custom = Pipeline([
    ('feature_eng', EngagementRatioTransformer()),
    ('prep',        extended_preprocessor),
    ('model',       LogisticRegression(max_iter=1000)),
])

pipeline_with_custom.fit(X_train, y_train)
auc_ext = roc_auc_score(y_test, pipeline_with_custom.predict_proba(X_test)[:,1])
print(f'Pipeline con feature engineering — AUC: {auc_ext:.4f} (vs {auc:.4f} base)')

## 3 — GridSearchCV sobre hiperparámetros del pipeline completo

In [ ]:
# Los nombres de parámetros siguen la forma: step__substep__param
param_grid = [
    {
        'model': [LogisticRegression(max_iter=1000)],
        'model__C': [0.01, 0.1, 1.0, 10.0],
    },
    {
        'model': [RandomForestClassifier(random_state=42)],
        'model__n_estimators': [50, 100],
        'model__max_depth': [3, 5, None],
    },
]

base_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', LogisticRegression()),  # placeholder — grid lo reemplaza
])

grid = GridSearchCV(
    base_pipeline, param_grid,
    cv=3, scoring='roc_auc', n_jobs=-1, verbose=0
)
grid.fit(X_train, y_train)

best_auc = roc_auc_score(y_test, grid.predict_proba(X_test)[:,1])
print(f'Mejor modelo: {type(grid.best_estimator_.named_steps["model"]).__name__}')
print(f'Parámetros:   {grid.best_params_}')
print(f'AUC test:     {best_auc:.4f}')

## 4 — Memory caching: evitar re-computar preprocesamiento (Géron Cap. 2)

In [ ]:
import time
import tempfile

cache_dir = tempfile.mkdtemp()

# Sin cache
pipe_nocache = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestClassifier(n_estimators=50, random_state=42)),
])

# Con cache: preprocesamiento se calcula 1 sola vez
pipe_cached = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestClassifier(n_estimators=50, random_state=42)),
], memory=cache_dir)

param_cache = {'model__max_depth': [3, 5, None]}

t0 = time.time()
GridSearchCV(pipe_nocache, param_cache, cv=3, scoring='roc_auc').fit(X_train, y_train)
t_nocache = time.time() - t0

t0 = time.time()
GridSearchCV(pipe_cached, param_cache, cv=3, scoring='roc_auc').fit(X_train, y_train)
t_cached = time.time() - t0

print(f'Sin cache:  {t_nocache:.2f}s')
print(f'Con cache:  {t_cached:.2f}s')
print(f'Speedup:    {t_nocache/t_cached:.1f}x (mayor con preprocesamiento costoso)')

## 5 — Serialización y deployment con joblib

In [ ]:
# Guardar el mejor pipeline
joblib.dump(grid.best_estimator_, 'data/best_pipeline.pkl', compress=3)

# Simular carga en producción
loaded_pipe = joblib.load('data/best_pipeline.pkl')

# Inferencia en producción: datos crudos → predicción
new_users = pd.DataFrame([{
    'sessions': 8, 'time_on_site': 300, 'pages': 4.5,
    'days_since_reg': 5, 'revenue_usd': 120.0,
    'channel': 'organic', 'device': 'mobile', 'plan': 'free'
}, {
    'sessions': 1, 'time_on_site': np.nan, 'pages': 1.2,
    'days_since_reg': 28, 'revenue_usd': np.nan,
    'channel': 'paid', 'device': 'desktop', 'plan': 'basic'
}])

probas = loaded_pipe.predict_proba(new_users)[:,1]
for i, p in enumerate(probas):
    print(f'Usuario {i+1}: P(activación) = {p:.3f} → {"ACTIVAR" if p > 0.5 else "no activar"}')

import os; print(f'\nModelo guardado: {os.path.getsize("data/best_pipeline.pkl")/1024:.0f} KB')

## Resumen

| Patrón | Qué hace | Referencia |
|---|---|---|
| `Pipeline([steps])` | Encadena transformaciones → evita data leakage | Géron Cap. 2 |
| `ColumnTransformer` | Transforma numéricas y categóricas en paralelo | Géron Cap. 2 |
| `BaseEstimator + TransformerMixin` | Custom transformer compatible con Pipeline/CV | Géron Cap. 2 |
| `GridSearchCV(pipeline, params)` | Accede a hiperparámetros con `step__param` | ESL 7.10 |
| `Pipeline(memory=dir)` | Cachea transformaciones costosas | Géron Cap. 2 |
| `joblib.dump/load` | Serialización para producción | Géron Cap. 2 |

**Siguiente:** `09_ensemble_methods.ipynb`